# Figure 5: developmental potential and bidirectional perturbations

Evaluate developmental-potential predictions across six datasets using saved concepts and CytoTRACE2 predictions. Visualize annotated and predicted potency, then perturb multipotent and differentiated programs in mouse neurogenesis.

Configure paths and analysis switches, then run the cells in order. Outputs are written to `outputs/figure5`.

## Prerequisites

- Install scConcept and prepare processed datasets using Tutorials 0–3.
- Generate the required concepts first, or use the deposited concept files in `Tutorial2/Concept`.
- Configure the input paths below. Processed `.mat` files use `Datasets/`; additional clinical, annotation, or plotting inputs are required by the corresponding analysis.

This tutorial starts from saved concepts. Run cells from top to bottom; preprocessing, topic training, and concept generation are not repeated. The shared input helpers are in `Tutorial2/tutorial_utils.py`.


In [ ]:
from pathlib import Path
import json
import gc
import hashlib
import platform
import importlib.metadata
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.io import loadmat
from scipy.stats import kendalltau
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
import scanpy as sc
import anndata as ad
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# =========================================================
# Locate the installed scConcept repository and tutorial inputs
# =========================================================
import os
import sys

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "scConcept.py").is_file() and (p / "Tutorial2").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the scConcept repository or one of its subdirectories.")
TUTORIAL_DIR = PROJECT_ROOT / "Tutorial2"
for directory in [PROJECT_ROOT, TUTORIAL_DIR]:
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from tutorial_utils import (ScConcept, SingleCellDataset, configure_inputs,
    input_path, dataset_path, load_mat_test, load_api_dataset, load_saved_concepts,
    score_saved_concepts, matlab_string, runtime_paths,
    compute_coherence_from_gene_lists, TD_eva_topics, read_pathways, interpretability)
PACKAGE_DATA_DIR = configure_inputs()
REPO_DIR = PROJECT_ROOT
CONCEPT_DIR = TUTORIAL_DIR / "Concept"

# Dataset paths can be overridden in Inputs/data_paths.local.json.
MAT_DIR = PACKAGE_DATA_DIR
LABEL_DIR = input_path("potency.labels", "potency/labels")
PLOT_INPUT_DIR = input_path("potency.plotting", "potency/plotting")
OUTPUT_DIR = TUTORIAL_DIR / "outputs/figure5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TOP_K = 100
RUN_POTENCY_UMAPS = True
RUN_PERTURBATION_UMAPS = True
RUN_CLASSIFIERS = True
RUN_RANDOM_GENE_CONTROLS = False
FOLDS = (8.0, 32.0, 64.0)
N_SPLITS = 10
N_RANDOM_GENE_SETS = 100
SPLIT_SEED = None
FORWARD_CLASSIFIER_SUPPRESSION = "reciprocal"
REVERSE_CLASSIFIER_SUPPRESSION = "reciprocal"
POTENCY_ORDER = ["Totipotent", "Pluripotent", "Multipotent", "Oligopotent", "Unipotent", "Differentiated"]
DATASETS = [
    ("Mouse_embryo_2_Smart_seq2_cytotrace2", "Mouse_embryo_2_Smart-seq2", "Mouse embryo 2 (Smart-seq2)"),
    ("Mouse_mature_neural_cell_types_10x_cytotrace2", "Mouse_mature_neural_cell_types_10x", "Mouse mature neural cell types (10x)"),
    ("Neural_crest_Smart_seq2_cytotrace2", "Neural_crest_Smart-seq2", "Neural crest (Smart-seq2)"),
    ("HSCs_and_MPPs_inDrop_cytotrace2", "HSCs_and_MPPs_inDrop", "HSCs and MPPs (inDrop)"),
    ("Retinal_neurons_10x_cytotrace2", "Retinal_neurons_10x", "Retinal neurons (10x)"),
    ("Mouse_neurogenesis_10x_cytotrace2", "Mouse_neurogenesis_10x", "Mouse neurogenesis (10x)"),
]


## 1. Load data and score potency concepts

Sum preprocessed expression over the first 100 listed genes per concept, without gene-wise standardization or division by gene-set size. Exclude potency categories with no matched genes from maximum-score assignment. Categories follow the six-level potency order above.

Compare predictions with `potency_annotation` using accuracy and Kendall's tau. A constant label vector has undefined tau and is reported as NaN. CytoTRACE2 predictions are read from the saved `CytoTRACE2_Potency` column.


In [ ]:



def load_potency_inputs(dataset, concept_name):
    mat_path = dataset_path(dataset)
    expression, genes, _ = load_mat_test(mat_path, dtype=np.float32)
    concepts = load_saved_concepts(CONCEPT_DIR / f"GPT5_{concept_name}_concepts.json")
    expected = ["Toti.", "Pluri.", "Multi.", "Oligo.", "Uni.", "Diff."]
    if [c["name"] for c in concepts] != expected:
        raise ValueError("Concept order must match the six potency categories.")
    truth = pd.read_csv(LABEL_DIR / f"{dataset}_cell_Potency.csv", index_col=0)
    cyto = pd.read_csv(LABEL_DIR / f"{dataset}_cell_Potency_CytoTRACE2.csv", index_col=0)
    if not truth.index.is_unique or not cyto.index.is_unique:
        raise ValueError("Potency table cell IDs must be unique.")
    if set(truth.index) != set(cyto.index):
        raise ValueError("Ground-truth and CytoTRACE2 cell IDs differ.")
    cyto = cyto.reindex(truth.index)
    if expression.shape != (len(truth), len(genes)):
        raise ValueError("Expression and potency labels have incompatible dimensions.")
    for labels in [truth["potency_annotation"], cyto["CytoTRACE2_Potency"]]:
        if labels.isna().any() or not set(labels).issubset(POTENCY_ORDER):
            raise ValueError("Missing or unknown potency labels.")
    return expression, genes, concepts, truth, cyto, mat_path


def potency_scores(expression, genes, concepts):
    gene_index = {g: i for i, g in enumerate(genes)}
    scores = np.zeros((expression.shape[0], len(concepts)), dtype=float)
    matched = []
    for i, concept in enumerate(concepts):
        idx = [gene_index[g] for g in concept["genes"][:TOP_K] if g in gene_index]
        matched.append(len(idx))
        if idx:
            block = expression[:, idx]
            block = block.toarray() if sparse.issparse(block) else np.asarray(block)
            scores[:, i] = block.astype(float).sum(axis=1)
    if not any(matched):
        raise ValueError("No concept genes match the expression input.")
    prediction_scores = scores.copy()
    prediction_scores[:, np.asarray(matched) == 0] = -np.inf
    prediction = np.asarray(POTENCY_ORDER)[prediction_scores.argmax(axis=1)]
    return scores, prediction, matched


def potency_metrics(truth, prediction):
    rank = {label: i + 1 for i, label in enumerate(POTENCY_ORDER)}
    tau, pvalue = kendalltau([rank[v] for v in prediction], [rank[v] for v in truth])
    return dict(Accuracy=float(accuracy_score(truth, prediction)), Kendall_tau=float(tau),
                Kendall_p=float(pvalue), F1_macro=float(f1_score(truth, prediction, average="macro")),
                F1_weighted=float(f1_score(truth, prediction, average="weighted")))


In [ ]:
def plot_all_outputs(adata, dir_out, datasetname):
    sc.set_figure_params(scanpy=True, fontsize=14)
    plt.rc('legend', fontsize=14)
    labels = ['Differentiated', 'Unipotent', 'Oligopotent', 'Multipotent', 'Pluripotent', 'Totipotent']
    colors = ['#5E4FA2', '#66C2A5', '#E6F598', '#FEE08B', '#F46D43', '#9E0142']
    potency_palette = dict(zip(labels, colors))
    adata.obs['CytoTRACE2_Potency'] = adata.obs['CytoTRACE2_Potency'].astype(str)
    fig = sc.pl.umap(adata, color='CytoTRACE2_Potency', palette=potency_palette, title='CytoTRACE 2', return_fig=True)
    ax = fig.axes[0]
    ax.legend_.set_title('Potency category')
    ax.spines[['right', 'top']].set_visible(False)
    ax.set_aspect(1 / ax.get_data_ratio())
    plt.savefig(dir_out + '/cytotrace2_potency_category_UMAP_' + datasetname + '.pdf', format='pdf', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    potency_order = {'Differentiated': 0, 'Unipotent': 1, 'Oligopotent': 2, 'Multipotent': 3, 'Pluripotent': 4, 'Totipotent': 5}
    adata.obs['potency_annotation_numeric'] = adata.obs['potency_annotation'].astype(str).map(potency_order).astype(float)
    labels = ['Differentiated', 'Unipotent', 'Oligopotent', 'Multipotent', 'Pluripotent', 'Totipotent']
    colors = ['#5E4FA2', '#66C2A5', '#E6F598', '#FEE08B', '#F46D43', '#9E0142']
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list('potency_cmap', list(zip(np.linspace(0, 1, len(colors)), colors)))
    sc.set_figure_params(scanpy=True, fontsize=14)
    plt.rc('legend', fontsize=14)
    fig = sc.pl.umap(adata, color='potency_annotation_numeric', cmap=cmap, vmin=0, vmax=5, title='Potency annotation', colorbar_loc=None, return_fig=True)
    ax = fig.axes[0]
    cbar = plt.colorbar(ax.collections[0], ax=ax, fraction=0.035)
    cbar.set_ticks([0, 1, 2, 3, 4, 5])
    cbar.set_ticklabels(labels)
    cbar.ax.set_title('Potency category', pad=10, loc='left')
    offset = matplotlib.transforms.ScaledTranslation(0, -1 / 12, fig.dpi_scale_trans)
    for label in cbar.ax.yaxis.get_majorticklabels():
        label.set_transform(label.get_transform() + offset)
    ax.spines[['right', 'top']].set_visible(False)
    ax.set_aspect(1 / ax.get_data_ratio())
    plt.savefig(dir_out + '/potency_annotation_UMAP_' + datasetname + '.pdf', format='pdf', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    potency_order = {'Differentiated': 0, 'Unipotent': 1, 'Oligopotent': 2, 'Multipotent': 3, 'Pluripotent': 4, 'Totipotent': 5}
    adata.obs['our_anno_numeric'] = adata.obs['our_anno'].astype(str).map(potency_order).astype(float)
    labels = ['Differentiated', 'Unipotent', 'Oligopotent', 'Multipotent', 'Pluripotent', 'Totipotent']
    colors = ['#5E4FA2', '#66C2A5', '#E6F598', '#FEE08B', '#F46D43', '#9E0142']
    cmap = matplotlib.colors.LinearSegmentedColormap.from_list('potency_cmap', list(zip(np.linspace(0, 1, len(colors)), colors)))
    sc.set_figure_params(scanpy=True, fontsize=14)
    plt.rc('legend', fontsize=14)
    fig = sc.pl.umap(adata, color='our_anno_numeric', cmap=cmap, vmin=0, vmax=5, title='Our annotation', colorbar_loc=None, return_fig=True)
    ax = fig.axes[0]
    cbar = plt.colorbar(ax.collections[0], ax=ax, fraction=0.035)
    cbar.set_ticks([0, 1, 2, 3, 4, 5])
    cbar.set_ticklabels(labels)
    cbar.ax.set_title('Potency category', pad=10, loc='left')
    offset = matplotlib.transforms.ScaledTranslation(0, -1 / 12, fig.dpi_scale_trans)
    for label in cbar.ax.yaxis.get_majorticklabels():
        label.set_transform(label.get_transform() + offset)
    ax.spines[['right', 'top']].set_visible(False)
    ax.set_aspect(1 / ax.get_data_ratio())
    plt.savefig(dir_out + '/scConcept_annotation_UMAP_' + datasetname + '.pdf', format='pdf', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)


## 2. Evaluate and visualize all six datasets

Use the same expression UMAP for ground truth, CytoTRACE2, and scConcept. Select 10,000 highly variable genes from the plotting H5AD input, then run PCA with `svd_solver="arpack"`, neighbors, and UMAP. Keep the six-category color scale and square plotting aspect. Metrics and predictions are saved separately for each dataset.


In [ ]:
metric_rows = []
neurogenesis = None
for dataset, concept_name, display_name in DATASETS:
    out_dir = OUTPUT_DIR / concept_name
    out_dir.mkdir(exist_ok=True)
    expression, genes, concepts, truth, cyto, mat_path = load_potency_inputs(dataset, concept_name)
    scores, scconcept_prediction, matched = potency_scores(expression, genes, concepts)
    cyto_prediction = cyto["CytoTRACE2_Potency"].astype(str).to_numpy()
    y_true = truth["potency_annotation"].astype(str).to_numpy()
    for method, prediction in [("scConcept", scconcept_prediction), ("CytoTRACE2", cyto_prediction)]:
        metric_rows.append(dict(dataset=display_name, dataset_id=concept_name, method=method,
                                n_cells=len(truth), **potency_metrics(y_true, prediction)))
    predictions = pd.DataFrame({"potency_annotation": y_true, "scConcept": scconcept_prediction,
                                 "CytoTRACE2": cyto_prediction}, index=truth.index)
    predictions.to_csv(out_dir / "predictions.csv")
    np.savez_compressed(out_dir / "potency_scores.npz", scores=scores, categories=np.asarray(POTENCY_ORDER),
                        matched_genes=np.asarray(matched), cell_ids=truth.index.to_numpy(dtype=str))
    if RUN_POTENCY_UMAPS:
        adata = sc.read_h5ad(PLOT_INPUT_DIR / f"{dataset}.h5ad")
        if set(adata.obs_names) != set(truth.index):
            raise ValueError(f"Plotting cell IDs do not match potency tables for {dataset}.")
        adata = adata[truth.index].copy()
        sc.pp.highly_variable_genes(adata, n_top_genes=10000)
        adata = adata[:, adata.var["highly_variable"]].copy()
        sc.tl.pca(adata, svd_solver="arpack")
        sc.pp.neighbors(adata)
        sc.tl.umap(adata)
        adata.obs["potency_annotation"] = y_true
        adata.obs["CytoTRACE2_Potency"] = cyto_prediction
        adata.obs["our_anno"] = scconcept_prediction
        with plt.rc_context():
            sc.set_figure_params(scanpy=True, fontsize=14, figsize=(4, 4))
            plot_all_outputs(adata, str(out_dir), concept_name)
        np.savez_compressed(out_dir / "expression_umap.npz", coordinates=adata.obsm["X_umap"],
                            cell_ids=adata.obs_names.to_numpy(dtype=str))
        if dataset == "Mouse_neurogenesis_10x_cytotrace2":
            neurogenesis = adata.copy()
        del adata
    if dataset == "Mouse_neurogenesis_10x_cytotrace2":
        neuro_concepts = concepts
        if neurogenesis is None:
            neurogenesis = sc.read_h5ad(PLOT_INPUT_DIR / f"{dataset}.h5ad")
            if set(neurogenesis.obs_names) != set(truth.index):
                raise ValueError("Neurogenesis cell IDs do not match potency tables.")
            neurogenesis = neurogenesis[truth.index].copy()
            sc.pp.highly_variable_genes(neurogenesis, n_top_genes=10000)
            neurogenesis = neurogenesis[:, neurogenesis.var["highly_variable"]].copy()
            neurogenesis.obs["potency_annotation"] = y_true
    del expression, scores
    gc.collect()
metrics = pd.DataFrame(metric_rows)
metrics.to_csv(OUTPUT_DIR / "potency_metrics.csv", index=False)
for _, concept_name, _ in DATASETS:
    metrics.loc[metrics.dataset_id == concept_name].to_csv(OUTPUT_DIR / concept_name / "metrics.csv", index=False)
display(metrics)


## 3. Bidirectional neurogenesis perturbations

Use all 34 multipotent genes and 44 differentiated genes. Project perturbed cells into a PCA/UMAP reference fitted to unperturbed multipotent followed by differentiated cells. Visualizations upregulate target genes by the configured fold and downregulate source genes by its reciprocal.

The perturbation figures swap the two UMAP coordinates for plotting, reverse the horizontal axis, and use an 8×8 scatter/KDE layout. Forward and reverse directions retain their separate palettes.


In [ ]:
by_name = {c["name"]: c for c in neuro_concepts}
multipotent_genes = by_name["Multi."]["genes"][:TOP_K]
differentiated_genes = by_name["Diff."]["genes"][:TOP_K]
assert len(multipotent_genes) == 34 and len(differentiated_genes) == 44
neurogenesis.X = neurogenesis.X.toarray() if sparse.issparse(neurogenesis.X) else np.asarray(neurogenesis.X)
multi_mask = neurogenesis.obs["potency_annotation"].astype(str).to_numpy() == "Multipotent"
diff_mask = neurogenesis.obs["potency_annotation"].astype(str).to_numpy() == "Differentiated"
if not multi_mask.any() or not diff_mask.any():
    raise ValueError("Both multipotent and differentiated cells are required.")
X_multi = neurogenesis.X[multi_mask]
X_diff = neurogenesis.X[diff_mask]
reference_X = np.vstack([X_multi, X_diff])
reference_labels = ["Control"] * len(X_multi) + ["Stimulated"] * len(X_diff)
neuro_gene_index = {str(g): i for i, g in enumerate(neurogenesis.var_names)}
perturbation_out = OUTPUT_DIR / "Mouse_neurogenesis_10x/perturbations"
perturbation_out.mkdir(exist_ok=True)


def perturb_expression(expression, add_genes, remove_genes, fold, suppression="reciprocal"):
    result = np.array(expression, copy=True)
    add_idx = [neuro_gene_index[g] for g in add_genes if g in neuro_gene_index]
    remove_idx = [neuro_gene_index[g] for g in remove_genes if g in neuro_gene_index]
    if not add_idx or not remove_idx:
        raise ValueError("Both perturbation gene sets must match the expression input.")
    result[:, add_idx] *= fold
    if suppression == "zero":
        result[:, remove_idx] = 0
    elif suppression == "reciprocal":
        result[:, remove_idx] /= fold
    else:
        raise ValueError("Suppression must be zero or reciprocal.")
    return result

DIRECTIONS = {
    "forward": dict(source=X_multi, target=X_diff, add=differentiated_genes,
                    remove=multipotent_genes, target_name="Differentiated",
                    suppression=FORWARD_CLASSIFIER_SUPPRESSION),
    "reverse": dict(source=X_diff, target=X_multi, add=multipotent_genes,
                    remove=differentiated_genes, target_name="Multipotent",
                    suppression=REVERSE_CLASSIFIER_SUPPRESSION),
}
matched_genes = {name: {"requested": genes, "matched": [g for g in genes if g in neuro_gene_index],
                       "missing": [g for g in genes if g not in neuro_gene_index]}
                 for name, genes in [("Multipotent", multipotent_genes), ("Differentiated", differentiated_genes)]}
(perturbation_out / "perturbation_genes.json").write_text(json.dumps(matched_genes, indent=2))


In [ ]:
def plot_forward_transition(reference_umap, projected, reference_labels, fold, out_dir, direction):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    all_umap = np.vstack([reference_umap, projected])
    adata_visualization = ad.AnnData(all_umap)
    adata_visualization.obsm['X_umap'] = all_umap
    adata_visualization.obs['batch'] = reference_labels + ['In silico stimulated'] * len(projected)
    plt.rcdefaults()
    sns.reset_defaults()
    sns.set_style('white')
    hue_order = ['Control', 'Stimulated', 'In silico stimulated']
    palette = {'Control': '#FEE08B', 'Stimulated': '#5E4FA2', 'In silico stimulated': '#66C2A5'}
    data_control = pd.DataFrame({'UMAP1': adata_visualization.obsm['X_umap'][np.array(adata_visualization.obs['batch']) == 'Control', 1], 'UMAP2': adata_visualization.obsm['X_umap'][np.array(adata_visualization.obs['batch']) == 'Control', 0], 'cell_type': 'Control'})
    data_stimulated = pd.DataFrame({'UMAP1': adata_visualization.obsm['X_umap'][np.array(adata_visualization.obs['batch']) == 'Stimulated', 1], 'UMAP2': adata_visualization.obsm['X_umap'][np.array(adata_visualization.obs['batch']) == 'Stimulated', 0], 'cell_type': 'Stimulated'})
    data_in_silico = pd.DataFrame({'UMAP1': adata_visualization.obsm['X_umap'][np.array(adata_visualization.obs['batch']) == 'In silico stimulated', 1], 'UMAP2': adata_visualization.obsm['X_umap'][np.array(adata_visualization.obs['batch']) == 'In silico stimulated', 0], 'cell_type': 'In silico stimulated'})
    data = pd.concat([data_control, data_stimulated, data_in_silico], ignore_index=True)
    fig = plt.figure(figsize=(8, 8))
    gs = fig.add_gridspec(2, 1, height_ratios=[1, 4])
    ax_scatter = fig.add_subplot(gs[1])
    sns.scatterplot(data=data, x='UMAP1', y='UMAP2', hue='cell_type', hue_order=hue_order, palette=palette, s=15, alpha=0.8, edgecolor='none', legend=False, ax=ax_scatter)
    ax_scatter.invert_xaxis()
    ax_scatter.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax_scatter.set_xlabel('')
    ax_scatter.set_ylabel('')
    ax_scatter.spines['top'].set_visible(False)
    ax_scatter.spines['right'].set_visible(False)
    ax_kde = fig.add_subplot(gs[0], sharex=ax_scatter)
    sns.kdeplot(data=data, x='UMAP1', hue='cell_type', hue_order=hue_order, palette=palette, fill=True, linewidth=2.0, common_norm=False, legend=False, ax=ax_kde, alpha=0.5)
    ax_kde.set_axis_off()
    plt.tight_layout()
    plt.savefig(str(out_dir / f'{direction}_{fold:g}x.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)


In [ ]:
def plot_reverse_transition(reference_umap, projected, reference_labels, fold, out_dir, direction):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    import pandas as pd
    all_umap = np.vstack([reference_umap, projected])
    adata_visualization = ad.AnnData(all_umap)
    adata_visualization.obsm['X_umap'] = all_umap
    adata_visualization.obs['batch'] = reference_labels + ['In silico stimulated'] * len(projected)
    hue_order = ['Control', 'Stimulated', 'In silico stimulated']
    palette = {'Control': '#FEE08B', 'Stimulated': '#66C2A5', 'In silico stimulated': '#5E4FA2'}
    data_control = pd.DataFrame({'UMAP1': adata_visualization.obsm['X_umap'][np.array(adata_visualization.obs['batch']) == 'Control', 1], 'UMAP2': adata_visualization.obsm['X_umap'][np.array(adata_visualization.obs['batch']) == 'Control', 0], 'cell_type': 'Control'})
    data_stimulated = pd.DataFrame({'UMAP1': adata_visualization.obsm['X_umap'][np.array(adata_visualization.obs['batch']) == 'Stimulated', 1], 'UMAP2': adata_visualization.obsm['X_umap'][np.array(adata_visualization.obs['batch']) == 'Stimulated', 0], 'cell_type': 'Stimulated'})
    data_in_silico = pd.DataFrame({'UMAP1': adata_visualization.obsm['X_umap'][np.array(adata_visualization.obs['batch']) == 'In silico stimulated', 1], 'UMAP2': adata_visualization.obsm['X_umap'][np.array(adata_visualization.obs['batch']) == 'In silico stimulated', 0], 'cell_type': 'In silico stimulated'})
    data = pd.concat([data_control, data_stimulated, data_in_silico], ignore_index=True)
    fig = plt.figure(figsize=(8, 8))
    gs = fig.add_gridspec(2, 1, height_ratios=[1, 4])
    ax_scatter = fig.add_subplot(gs[1])
    sns.scatterplot(data=data, x='UMAP1', y='UMAP2', hue='cell_type', hue_order=hue_order, palette=palette, s=15, alpha=0.8, edgecolor='none', legend=False, ax=ax_scatter)
    ax_scatter.set_xlabel('')
    ax_scatter.set_ylabel('')
    ax_scatter.invert_xaxis()
    ax_scatter.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax_scatter.spines['top'].set_visible(False)
    ax_scatter.spines['right'].set_visible(False)
    ax_kde = fig.add_subplot(gs[0], sharex=ax_scatter)
    sns.kdeplot(data=data, x='UMAP1', hue='cell_type', hue_order=hue_order, palette=palette, linewidth=2.0, common_norm=False, ax=ax_kde, legend=False, fill=True, alpha=0.5)
    ax_kde.set_axis_off()
    plt.tight_layout()
    plt.savefig(str(out_dir / f'{direction}_{fold:g}x.pdf'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)


In [ ]:
if RUN_PERTURBATION_UMAPS:
    import umap
    pca = PCA(n_components=50, random_state=0)
    reference_pca = pca.fit_transform(reference_X)
    umap_model = umap.UMAP(n_neighbors=15, min_dist=0.5, random_state=42, verbose=True)
    reference_umap = umap_model.fit_transform(reference_pca)
    for direction, settings in DIRECTIONS.items():
        plot_function = plot_forward_transition if direction == "forward" else plot_reverse_transition
        for fold in FOLDS:
            perturbed = perturb_expression(settings["source"], settings["add"], settings["remove"], fold)
            projected = umap_model.transform(pca.transform(perturbed))
            plot_function(reference_umap, projected, reference_labels, fold, perturbation_out, direction)
            np.savez_compressed(perturbation_out / f"{direction}_{fold:g}x_coordinates.npz",
                                reference=reference_umap, projected=projected, labels=np.asarray(reference_labels))


## 4. Held-out perturbation classification

For each direction, fit logistic regression on unperturbed source and target cells using stratified 70:30 splits. Perturb only held-out source cells and report mean target-state probability over 10 runs. Save individual runs and mean/standard deviation at each strength.

Both directions downregulate source-concept genes by 1/fold, matching the projection experiments. `SPLIT_SEED=None` uses unseeded splits; set an integer to fix split randomness. Optional random-gene controls use 100 gene sets of matching sizes, sampled with seed 0, with one classifier split per random set.


In [ ]:
def classifier_runs(direction, settings):
    expression = np.vstack([settings["source"], settings["target"]])
    labels = np.r_[np.zeros(len(settings["source"]), dtype=int), np.ones(len(settings["target"]), dtype=int)]
    rows = []
    for run in range(N_SPLITS):
        seed = None if SPLIT_SEED is None else SPLIT_SEED + run
        train, test = train_test_split(np.arange(len(labels)), test_size=0.3, stratify=labels, random_state=seed)
        classifier = LogisticRegression(solver="liblinear")
        classifier.fit(expression[train], labels[train])
        heldout = expression[test[labels[test] == 0]]
        baseline = classifier.predict_proba(heldout)[:, 1]
        for fold in FOLDS:
            perturbed = perturb_expression(heldout, settings["add"], settings["remove"], fold, settings["suppression"])
            probability = classifier.predict_proba(perturbed)[:, 1]
            rows.append(dict(direction=direction, target=settings["target_name"], run=run, fold=fold,
                             suppression=settings["suppression"], mean_probability=float(probability.mean()),
                             baseline_probability=float(baseline.mean()),
                             test_accuracy=float(classifier.score(expression[test], labels[test])),
                             test_auc=float(roc_auc_score(labels[test], classifier.predict_proba(expression[test])[:, 1]))))
    return rows

if RUN_CLASSIFIERS:
    rows = []
    for direction, settings in DIRECTIONS.items():
        rows.extend(classifier_runs(direction, settings))
    classifier_results = pd.DataFrame(rows)
    classifier_results.to_csv(perturbation_out / "classifier_runs.csv", index=False)
    classifier_summary = classifier_results.groupby(["direction", "fold", "suppression"])["mean_probability"].agg(
        mean="mean", std=lambda values: np.std(values, ddof=0))
    classifier_summary.to_csv(perturbation_out / "classifier_summary.csv")
    display(classifier_summary)

if RUN_RANDOM_GENE_CONTROLS:
    rows = []
    for direction, settings in DIRECTIONS.items():
        rng = np.random.default_rng(0)
        for permutation in range(N_RANDOM_GENE_SETS):
            random_settings = dict(settings)
            random_settings["add"] = rng.choice(neurogenesis.var_names, len(settings["add"]), replace=False).tolist()
            random_settings["remove"] = rng.choice(neurogenesis.var_names, len(settings["remove"]), replace=False).tolist()
            expression = np.vstack([settings["source"], settings["target"]])
            labels = np.r_[np.zeros(len(settings["source"]), dtype=int), np.ones(len(settings["target"]), dtype=int)]
            train, test = train_test_split(np.arange(len(labels)), test_size=0.3, stratify=labels)
            classifier = LogisticRegression(solver="liblinear").fit(expression[train], labels[train])
            heldout = expression[test[labels[test] == 0]]
            for fold in FOLDS:
                perturbed = perturb_expression(heldout, random_settings["add"], random_settings["remove"], fold, settings["suppression"])
                rows.append(dict(direction=direction, random_set=permutation, fold=fold,
                                 mean_probability=float(classifier.predict_proba(perturbed)[:, 1].mean())))
    pd.DataFrame(rows).to_csv(perturbation_out / "random_gene_controls.csv", index=False)


## Output guide

Paths below are relative to `outputs/figure5`. Each dataset has its own output directory.

| Analysis | Main output |
|---|---|
| Six-dataset scConcept and CytoTRACE2 performance | `potency_metrics.csv` |
| Per-dataset predictions and metrics | `<dataset>/predictions.csv`, `<dataset>/metrics.csv` |
| Ground-truth potency UMAP | `<dataset>/potency_annotation_UMAP_<dataset>.pdf` |
| scConcept potency UMAP | `<dataset>/scConcept_annotation_UMAP_<dataset>.pdf` |
| CytoTRACE2 potency UMAP | `<dataset>/cytotrace2_potency_category_UMAP_<dataset>.pdf` |
| Potency scores and expression UMAP coordinates | `<dataset>/potency_scores.npz`, `<dataset>/expression_umap.npz` |
| Multipotent-to-differentiated perturbation | `Mouse_neurogenesis_10x/perturbations/forward_<fold>x.pdf` |
| Differentiated-to-multipotent perturbation | `Mouse_neurogenesis_10x/perturbations/reverse_<fold>x.pdf` |
| Held-out bidirectional classification | `Mouse_neurogenesis_10x/perturbations/classifier_runs.csv`, `Mouse_neurogenesis_10x/perturbations/classifier_summary.csv` |
| Optional random-gene controls | `Mouse_neurogenesis_10x/perturbations/random_gene_controls.csv` |

`<dataset>` is the dataset identifier in `DATASETS`; `<fold>` is the perturbation strength. Outputs depend on the enabled analysis switches. The final cell saves the run configuration and package versions.


## 5. Save run settings

Record input paths, concept checksums, scoring and perturbation parameters, and software versions.


In [ ]:
versions = {name: importlib.metadata.version(name) for name in ["numpy", "pandas", "scipy", "scikit-learn", "scanpy", "anndata", "umap-learn", "matplotlib", "seaborn"]}
configuration = dict(python=platform.python_version(), packages=versions,
    mat_dir=str(MAT_DIR), label_dir=str(LABEL_DIR), plot_input_dir=str(PLOT_INPUT_DIR), top_k=TOP_K,
    folds=list(FOLDS), n_splits=N_SPLITS, split_seed=SPLIT_SEED,
    classifier_suppression=dict(forward=FORWARD_CLASSIFIER_SUPPRESSION, reverse=REVERSE_CLASSIFIER_SUPPRESSION),
    concept_sha256={name: hashlib.sha256((CONCEPT_DIR / f"GPT5_{name}_concepts.json").read_bytes()).hexdigest()
                    for _, name, _ in DATASETS},
    enabled=dict(potency_umaps=RUN_POTENCY_UMAPS, perturbation_umaps=RUN_PERTURBATION_UMAPS,
                 classifiers=RUN_CLASSIFIERS, random_gene_controls=RUN_RANDOM_GENE_CONTROLS))
configuration["package_paths"] = runtime_paths()
(OUTPUT_DIR / "run_configuration.json").write_text(json.dumps(configuration, indent=2))
print("Outputs:", OUTPUT_DIR)
